# Junior cricket talent identification with PCA and Random Forests

This notebook is the main pipeline from my MSc Data Science project at Durham University, which I ran alongside the Durham County Cricket Academy Pathway. The aim was to rank every Under-13 cricketer in the county using their batting, bowling and fielding records from the 2025 Play-Cricket season, rather than relying on raw totals like runs or wickets on their own.

**A note on the data:** the original data is about children, so every player name and club has been replaced with an anonymous ID (e.g. `Player_005`, `Club_14`) before publishing. The numbers themselves are unchanged.

The pipeline runs in five steps:
1. Load the three Play-Cricket exports and merge them on player and club
2. Build a latent skill score for each discipline using PCA, oriented with "anchor" stats so that higher always means better
3. Combine the three scores with equal 33/33/33 weighting into an all-rounder target
4. Train a Random Forest on the raw features to predict that target, tuned with randomised search and cross-validation
5. Produce leaderboards and permutation feature importances

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

DATA_DIR = "../data"
FIG_DIR = "../figures"
OUT_DIR = "../outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

Batting = f"{DATA_DIR}/batting_statistics.csv"     # batting
Bowling = f"{DATA_DIR}/bowling_statistics.csv"     # bowling
Fielding = f"{DATA_DIR}/fielding_statistics.csv"   # fielding
TOP_K = 30 # top number of players (this can be edited)
RANDOM_STATE = 45

## Helpers: loading, merging and cleaning

Play-Cricket exports come as three separate CSVs, so each one gets a suffix (`_bat`, `_bowl`, `_field`) before an outer merge, which means a specialist batter with no bowling record still stays in the dataset. `BEST BOWLING` (e.g. `6/24`) is split into two numeric columns, and missing stats are filled with zero, since in this data a blank almost always means "didn't do it" rather than "wasn't recorded".

In [2]:
ID_COLS = {"Player", "Club"}
SPECIAL_BOWL = {"BEST BOWLING"}

# Columns produced by the pipeline that must NEVER feed back into the RF
MODEL_MADE_COLS = {
    "final_score_model", "bat_score", "bowl_score", "field_score", "allrounder_target"
}

RF_EXCLUDED_BASENAMES = {"rank"}  # exact base-name (case-insensitive) exclusion for model
SUFFIXES = ("_bat", "_bowl", "_field")

def strip_suffix(col: str) -> str:
    for s in SUFFIXES:
        if col.endswith(s):
            return col[: -len(s)]
    return col

def is_excluded_for_model(col: str) -> bool:
    base = strip_suffix(col).strip().lower()
    return (base in RF_EXCLUDED_BASENAMES) or (col in MODEL_MADE_COLS)

def parse_best_bowling(series: pd.Series) -> pd.DataFrame:
    # Parse 'BEST BOWLING' like '6/24' -> best_wkts=6, best_runs=24
    wkts, runs = [], []
    for v in series.fillna("0/0").astype(str):
        m = re.match(r"\s*(\d+)\s*/\s*(\d+)\s*", v)
        if m:
            wkts.append(float(m.group(1))); runs.append(float(m.group(2)))
        else:
            wkts.append(0.0); runs.append(0.0)
    return pd.DataFrame({"best_wkts": wkts, "best_runs": runs})

def add_suffix(df: pd.DataFrame, suffix: str, parse_best=False) -> pd.DataFrame:
    # Keep IDs, numeric-coerce others, optionally parse BEST BOWLING, then suffix non-ID columns.
    df = df.copy()
    # Coerce numerics except IDs + BEST BOWLING (string)
    for c in df.columns:
        if c not in ID_COLS | SPECIAL_BOWL:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # Parse BEST BOWLING into best_wkts/best_runs
    if parse_best and "BEST BOWLING" in df.columns:
        best_df = parse_best_bowling(df["BEST BOWLING"])
        df = pd.concat([df.drop(columns=["BEST BOWLING"]), best_df], axis=1)
    # Apply suffix to non-ID columns
    rename_map = {c: f"{c}{suffix}" for c in df.columns if c not in ID_COLS}
    df = df.rename(columns=rename_map)
    return df

def load_and_merge(path_bat, path_bowl, path_field=None) -> pd.DataFrame:
    bat = pd.read_csv(path_bat)
    bowl = pd.read_csv(path_bowl)
    bat = add_suffix(bat, "_bat", parse_best=False)
    bowl = add_suffix(bowl, "_bowl", parse_best=True)  # creates best_wkts_bowl & best_runs_bowl
    dfs = [bat, bowl]
    if path_field:
        field = pd.read_csv(path_field)
        field = add_suffix(field, "_field", parse_best=False)
        dfs.append(field)
    # Outer merge on IDs
    merged = dfs[0]
    for nxt in dfs[1:]:
        merged = pd.merge(merged, nxt, on=list(ID_COLS), how="outer")
    # Safe numeric NA handling
    num_cols = merged.select_dtypes(include=[np.number]).columns
    merged[num_cols] = merged[num_cols].fillna(0)
    return merged

def minmax01(x: np.ndarray):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return np.zeros_like(x) if mx - mn == 0 else (x - mn) / (mx - mn)

def split_feature_groups(df: pd.DataFrame):
    # Return X_bat, X_bowl, X_field using *_bat/_bowl/_field suffixes,
    # but EXCLUDE any ranking columns and model-made columns from the RF features.
    
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    # Discipline-specific
    bat_cols   = [c for c in numeric_cols if c.endswith("_bat")]
    bowl_cols  = [c for c in numeric_cols if c.endswith("_bowl")]
    field_cols = [c for c in numeric_cols if c.endswith("_field")]
    # DataFrames for latent learning (keep everything within the discipline blocks)
    X_bat   = df[bat_cols].astype(float).copy()   if bat_cols   else pd.DataFrame(index=df.index)
    X_bowl  = df[bowl_cols].astype(float).copy()  if bowl_cols  else pd.DataFrame(index=df.index)
    X_field = df[field_cols].astype(float).copy() if field_cols else pd.DataFrame(index=df.index)
    # RF feature pool across all disciplines, with exclusions
    X_all_cols = sorted(set(bat_cols) | set(bowl_cols) | set(field_cols))
    X_all_cols = [c for c in X_all_cols if not is_excluded_for_model(c)]
    X_all  = df[X_all_cols].astype(float).copy()  if X_all_cols else pd.DataFrame(index=df.index)
    return X_bat, X_bowl, X_field, X_all

def learn_latent(x_df: pd.DataFrame, anchors_pos=None, anchors_neg=None, random_state=42):
    # Standardise -> PCA(1) -> orient sign with anchors (pos increase score, neg decrease score)
    if x_df.empty:
        return np.zeros(len(x_df), dtype=float)
    Xz = StandardScaler().fit_transform(x_df.values)
    z1 = PCA(n_components=1, random_state=random_state).fit_transform(Xz).ravel()
    flip_score = 0
    if anchors_pos:
        for a in anchors_pos:
            if a in x_df.columns:
                r = np.corrcoef(z1, x_df[a].values)[0, 1]
                if np.isfinite(r): flip_score += np.sign(r)
    if anchors_neg:
        for a in anchors_neg:
            if a in x_df.columns:
                r = np.corrcoef(z1, x_df[a].values)[0, 1]
                if np.isfinite(r): flip_score -= np.sign(r)
    if flip_score < 0:
        z1 = -z1
    return z1

def prune_collinearity(X: pd.DataFrame, thr=0.95):
    if X.shape[1] <= 1:
        return X
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > thr)]
    kept = [c for c in X.columns if c not in to_drop]
    return X[kept].copy()

## Latent skill scores and the Random Forest

For each discipline the features are standardised and reduced to their first principal component. Because the sign of a PCA component is arbitrary, I correlate PC1 with a set of anchor stats (runs, wickets, catches as positives; ducks, economy rate as negatives) and flip it if needed. The three scores are min-max scaled to [0, 1] and averaged equally, because I wanted to see what happened when batting wasn't given its usual head start.

A Random Forest is then trained on the full feature set (with near-duplicate columns above 0.95 correlation pruned), tuned by randomised search over 50 combinations with 5-fold cross-validation to minimise MAE, and checked on a 25% hold-out set.

In [3]:
def train_rf_self_supervised_equal_thirds(df: pd.DataFrame, random_state=RANDOM_STATE):
    X_bat, X_bowl, X_field, X_all = split_feature_groups(df)
    if X_all.empty:
        df["final_score_model"] = 0.0
        return df, pd.Series(dtype=float), {}, {}

    # Latent targets 
    bat_latent = learn_latent(
        X_bat,
        anchors_pos=[c for c in X_bat.columns if any(k in c for k in ["RUNS", "AVG", "50s", "100s", "STRIKE RATE", "4s", "6s", "NOT OUTS"])],
        anchors_neg=[c for c in X_bat.columns if "DUCKS" in c],
        random_state=random_state
    ) if not X_bat.empty else np.zeros(len(df))

    bowl_latent = learn_latent(
        X_bowl,
        anchors_pos=[c for c in X_bowl.columns if any(k in c for k in ["WICKETS", "MAIDENS", "OVERS", "5 WICKET HAUL", "best_wkts"])],
        anchors_neg=[c for c in X_bowl.columns if any(k in c for k in ["ECONOMY RATE", "AVERAGE", "STRIKE RATE", "RUNS_bowl", "best_runs"])],
        random_state=random_state
    ) if not X_bowl.empty else np.zeros(len(df))

    field_latent = learn_latent(
        X_field,
        anchors_pos=[c for c in X_field.columns if any(k in c for k in ["CAUGHT", "STUMPED", "RUN OUT", "DISMISS", "CATCH"])],
        anchors_neg=None,
        random_state=random_state
    ) if not X_field.empty else np.zeros(len(df))

    # Scale [0,1]
    bat_scaled   = minmax01(bat_latent)
    bowl_scaled  = minmax01(bowl_latent)
    field_scaled = minmax01(field_latent)

    # 3weights target into 3rds (33% each)
    y_target = (bat_scaled + bowl_scaled + field_scaled) / 3.0
    component_scores = {
        "bat_score": bat_scaled,
        "bowl_score": bowl_scaled,
        "field_score": field_scaled,
        "allrounder_target": y_target
    }

    # Features (prune collinearity at the 95% threshold) 
    X = prune_collinearity(X_all, thr=0.95)

    # RF with hyperparam search (minimize MAE)
    rf = RandomForestRegressor(n_jobs=-1, random_state=random_state)
    param_dist = {
        "n_estimators": [600, 800, 1000, 1200],
        "max_depth": [None, 12, 20, 28, 36],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8],
        "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
        "bootstrap": [True],
        "max_samples": [None, 0.7, 0.85]
    }
    cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
    mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
    search = RandomizedSearchCV(
        rf, param_dist, n_iter=50, scoring=mae_scorer,
        cv=cv, random_state=random_state, n_jobs=-1, verbose=0
    )
    search.fit(X, y_target)
    best_rf = search.best_estimator_

    # Fit on full data & predict
    best_rf.fit(X, y_target)
    df["final_score_model"] = best_rf.predict(X)

    # Holdout sanity check
    X_tr, X_te, y_tr, y_te = train_test_split(X, y_target, test_size=0.25, random_state=random_state)
    best_rf.fit(X_tr, y_tr)
    pred = best_rf.predict(X_te)
    holdout_metrics = {"r2": float(r2_score(y_te, pred)), "mae": float(mean_absolute_error(y_te, pred))}
    print(f"[Holdout] R^2: {holdout_metrics['r2']:.3f} | MAE: {holdout_metrics['mae']:.4f}")

    # Permutation importances on full data
    best_rf.fit(X, y_target)
    perm = permutation_importance(best_rf, X, y_target, n_repeats=10, random_state=random_state, n_jobs=-1)
    perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

    # Pred vs Actual chart (+saved)
    plt.figure(figsize=(7, 6))
    plt.scatter(y_te, pred, alpha=0.7, marker='o', edgecolors="k", linewidths=0.5)
    plt.plot([0, 1], [0, 1], "--", linewidth=2, label="Perfect Prediction")
    plt.xlim(0, 1); plt.ylim(0, 1)
    plt.xlabel("Actual All-Rounder Target Score")
    plt.ylabel("Predicted Score (Random Forest)")
    plt.title("Random Forest Accuracy: Predicted vs Actual")
    plt.legend()
    plt.text(0.05, 0.92, f"[Holdout] R^2: {holdout_metrics['r2']:.3f} | MAE: {holdout_metrics['mae']:.4f}",
             fontsize=11, bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/rf_pred_vs_actual.png", dpi=300)
    plt.show()
    print("Figure saved as rf_pred_vs_actual.png")

    # Best parameters from the search above
    best_params = search.best_params_
    print("\n[Best Hyperparameters Found]")
    for k, v in best_params.items():
        print(f"{k}: {v}")

    return df, perm_imp, component_scores, holdout_metrics

## Reporting: leaderboards and feature importance

In [4]:
def pretty_top(df: pd.DataFrame, cols: list, k: int, title: str):
    print(f"\n{title}")
    print(df.loc[:, cols].head(k).to_string(index=False))

# shows feature importances from RF
def plot_importances(importances: pd.Series, top_n=10):
    if importances is None or len(importances) == 0:
        print("No feature importances to plot."); return
    top_imp = importances.head(top_n).sort_values(ascending=True)
    n = len(top_imp)
    cmap = plt.get_cmap("tab20", n)
    colors = [cmap(i) for i in range(n)]
    plt.figure(figsize=(9, 6))
    bars = plt.barh(top_imp.index, top_imp.values, color=colors)
    plt.xlabel("Permutation importance (mean)")
    plt.title(f"Top {top_n} Feature Importances — Self-Supervised RF (33/33/33)")
    plt.tight_layout()
    for bar, val in zip(bars, top_imp.values):
        plt.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                 f"{val:.4f}", va="center", ha="left", fontsize=9)
    plt.savefig(f"{FIG_DIR}/feature_importances_top10.png", dpi=300)
    plt.show()
    print("Figure saved as feature_importances_top10.png")

## Run the pipeline

In [5]:
def main():
    df = load_and_merge(Batting, Bowling, Fielding)

    # Train the self-supervised RF with 33%/33%/33% target
    df_model, importances, parts, holdout = train_rf_self_supervised_equal_thirds(df.copy(), random_state=RANDOM_STATE)

    # Attach component scores for reporting/ranking
    for k, v in parts.items():
        df_model[k] = v

    # Leaderboards
    ranked_all = df_model.sort_values("final_score_model", ascending=False).reset_index(drop=True)
    ranked_bat = df_model.sort_values("bat_score", ascending=False).reset_index(drop=True)
    ranked_bowl = df_model.sort_values("bowl_score", ascending=False).reset_index(drop=True)
    ranked_field = df_model.sort_values("field_score", ascending=False).reset_index(drop=True)

    # Output columns — include best_wkts/best_runs if present
    cols_out = ["Player", "Club", "final_score_model", "bat_score", "bowl_score", "field_score"]
    for extra in ["best_wkts_bowl", "best_runs_bowl"]:
        if extra in df_model.columns:
            cols_out.append(extra)

    pretty_top(ranked_all, cols_out, TOP_K, f"Top {TOP_K} — All-Rounder (RF on 33/33/33 target)")
    pretty_top(ranked_bat, ["Player", "Club", "bat_score"], TOP_K, f"Top {TOP_K} — Best Batters (latent)")
    pretty_top(ranked_bowl, ["Player", "Club", "bowl_score", "best_wkts_bowl", "best_runs_bowl"] if "best_wkts_bowl" in df_model.columns else ["Player", "Club", "bowl_score"],
               TOP_K, f"Top {TOP_K} — Best Bowlers (latent)")
    pretty_top(ranked_field, ["Player", "Club", "field_score"], TOP_K, f"Top {TOP_K} — Best Fielders (latent)")

    # Save CSVs
    ranked_all.head(TOP_K).to_csv(f"{OUT_DIR}/top_allrounders_rf_33_33_33.csv", index=False)
    ranked_bat.head(TOP_K).to_csv(f"{OUT_DIR}/top_batters_latent.csv", index=False)
    ranked_bowl.head(TOP_K).to_csv(f"{OUT_DIR}/top_bowlers_latent.csv", index=False)
    ranked_field.head(TOP_K).to_csv(f"{OUT_DIR}/top_fielders_latent.csv", index=False)

    # Print importances + plot
    if importances is not None and len(importances) > 0:
        print("\n[Permutation Feature Importances]")
        print(importances.round(4).to_string())
    plot_importances(importances)

    # output the best by role
    best_batter = ranked_bat.loc[0, ["Player", "Club", "bat_score"]].to_dict() if not ranked_bat.empty else {}
    best_bowler = ranked_bowl.loc[0, ["Player", "Club", "bowl_score"]].to_dict() if not ranked_bowl.empty else {}
    best_fielder = ranked_field.loc[0, ["Player", "Club", "field_score"]].to_dict() if not ranked_field.empty else {}
    best_allrounder = ranked_all.loc[0, ["Player", "Club", "final_score_model"]].to_dict() if not ranked_all.empty else {}

    print("\n=== Best by Role ===")
    print("Batter:", best_batter)
    print("Bowler:", best_bowler)
    print("Fielder:", best_fielder)
    print("All-Rounder (model):", best_allrounder)

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[Holdout] R^2: 0.980 | MAE: 0.0177


Figure saved as rf_pred_vs_actual.png

[Best Hyperparameters Found]
n_estimators: 600
min_samples_split: 2
min_samples_leaf: 1
max_samples: 0.85
max_features: 0.5
max_depth: 12
bootstrap: True

Top 30 — All-Rounder (RF on 33/33/33 target)
    Player    Club  final_score_model  bat_score  bowl_score  field_score  best_wkts_bowl  best_runs_bowl
Player_005 Club_14           0.785396   0.811505    0.840968     0.875142             3.0            13.0
Player_316 Club_24           0.736883   0.710841    0.795950     0.787865             4.0            11.0
Player_379 Club_36           0.732928   0.608687    0.949874     0.716390             4.0            16.0
Player_421 Club_15           0.725714   1.000000    0.957200     0.439569             3.0             2.0
Player_247 Club_29           0.722516   0.762170    0.673416     0.796181             2.0             2.0
Player_152 Club_08           0.720490   0.624326    0.901527     0.746505             4.0            13.0
Player_149 Club_34 

Figure saved as feature_importances_top10.png

=== Best by Role ===
Batter: {'Player': 'Player_421', 'Club': 'Club_15', 'bat_score': 1.0}
Bowler: {'Player': 'Player_170', 'Club': 'Club_43', 'bowl_score': 1.0}
Fielder: {'Player': 'Player_124', 'Club': 'Club_48', 'field_score': 1.0}
All-Rounder (model): {'Player': 'Player_005', 'Club': 'Club_14', 'final_score_model': 0.7853956169268799}


## What I'd do differently next time

- **The all-rounder target is built from the same features the model is trained on**, so the very high hold-out R² mostly shows that the Random Forest can recover a score I constructed, rather than proving it predicts future performance. The more useful outputs are the feature importances and the leaderboards. A proper test would be to validate the rankings against real county selections or players' performance the following season.
- **Scorecard data has no match context.** Fifty runs against a strong attack on a seaming pitch counts the same as fifty against a weak attack on a flat one. Opposition strength, match situation and ball-by-ball data would make the rankings far more meaningful.
- **Fielding PC1 mixes outfielders and wicketkeepers**, which means keepers tend to score lower on fielding. Splitting keeping and outfield work would give a fairer fielding measure.
- **A few Play-Cricket columns are stored as text** (e.g. `GAMES WON(%)` as `12(63.16%)`), so they are converted to zero and carry no signal. Parsing them properly is an easy fix.
- **The Play-Cricket `Rank` column is included in the PCA step**, and since that rank is itself ordered by runs, wickets or victims it slightly double counts those stats. It is correctly excluded from the Random Forest features.
- **Zero imputation treats "not observed" the same as "no performance"**, which could disadvantage players who missed games through injury or selection.